In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from post_processing.additional.loaders import enrich_df_with_whole_who_tree_classification
from post_processing.column_normalizer import ColumnNormalizer
from post_processing.column_normalizer import SimplifiedColumnNormalizer
from post_processing.deduplication import DropDuplicates
from post_processing.droppers import DropNARowsInColumns
from post_processing.droppers import DropRowsWithValuesInColumn, DropIfNNAInRow, DropAllRowsThatHaveNotNumericSigns
from post_processing.droppers import DropRowsWithValuesInColumnExactMatch
from post_processing.expression_types_utils import FlipNegativeToPositive, MapExpressionTypes
from post_processing.group_utils import SelectHistologicalGroupFromTable
from post_processing.loaders import get_papers
from post_processing.loaders import path_with_content_root
from post_processing.math import CalculateExprPctAgain
from post_processing.math import CalculateMissingValues
from post_processing.merge_by_table import MergeDifferingByColumnsPerTableID
from post_processing.overwrites import ConditionalOverWrite
from post_processing.pipeline_utils import Pipeline
from post_processing.pipeline_utils import SubPipeline
from post_processing.string_cleaners import CleanColumn
from post_processing.string_cleaners import MergeDifferingByOnlyOneSign
from post_processing.tables_to_drop import TABLES_TO_DROP
from post_processing.tumour_mapper import TumourMapperV3
from post_processing.variations_dicts import PATTERN_INTO_PD_NAN
from post_processing.variations_dicts import PATTERN_VARIATIONS_DICT
from post_processing.variations_dicts import STAINS_VARIATIONS_DICTIONARY, EXPR_TYPE_VARIATIONS_DICT
from post_processing.variations_dicts import STAINS_VARIATIONS_DICTIONARY_FOR_REGEX

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# from post_processing.loaders import get_pairs
# from post_processing.loaders import path_with_content_root
# abstracts = get_pairs(path_with_content_root("data_retrieval/please_work_abstracts/"))
# tables = get_pairs(path_with_content_root("data_retrieval/please_work_tables/"),
#                    "*_2_5_flash.json",
#                    "table")
# print(tables.shape, tables.table_id.nunique(), abstracts.shape, abstracts.PMID.nunique())

# Loaded as an raw CSV file below.

In [3]:
PAPERS = get_papers("../database_versions/papers_metadata.csv")
PMID_TO_PT = PAPERS.set_index('PMID')['PT'].to_dict()
PAPERS.shape

(5961, 66)

In [4]:
VALUE_COLUMNS = ['total_N', 'expr_n', 'expr_pct']

df_to_check_columns = pd.read_csv("../database_versions/raw.csv",
                                  low_memory=False, nrows=2).reset_index(drop=True)

column_processing_map = {
    **{col: CleanColumn() for col in df_to_check_columns.columns if col not in VALUE_COLUMNS + ['tumour', 'stain']},
    "tumour": SubPipeline([CleanColumn()]),
    "stain": SubPipeline([CleanColumn(), MergeDifferingByOnlyOneSign('-')])
}

pipeline = Pipeline(column_processing_map,
                    [
                        ColumnNormalizer(variations_dict=STAINS_VARIATIONS_DICTIONARY,
                                         regex_variations_dict=STAINS_VARIATIONS_DICTIONARY_FOR_REGEX,
                                         key_column='stain'),
                        ColumnNormalizer(variations_dict=EXPR_TYPE_VARIATIONS_DICT,
                                         regex_variations_dict={},
                                         key_column='expr_type'),
                        SimplifiedColumnNormalizer(variations_dict=PATTERN_VARIATIONS_DICT,
                                                   key_column='pattern'),

                        # Dropping wrong paper types.
                        DropRowsWithValuesInColumn(column='PT',
                                                   values=['review', 'meta', 'systematic',
                                                           'editorial', 'letter',
                                                           'comment', 'news', 'interview',
                                                           'published erratum', 'retracted publication',
                                                           'retraction of publication'
                                                           ]),

                        # Dropping manually excluded tables.
                        DropRowsWithValuesInColumn(
                            column="table_id",
                            values=TABLES_TO_DROP
                        ),

                        DropNARowsInColumns(columns=['stain', 'tumour', 'expr_type', 'expr_n']),
                        DropIfNNAInRow(n=2, columns=['total_N', 'expr_n', 'expr_pct']),

                        DropAllRowsThatHaveNotNumericSigns(column='expr_pct'),
                        DropAllRowsThatHaveNotNumericSigns(column='total_N'),
                        DropAllRowsThatHaveNotNumericSigns(column='expr_n'),

                        DropRowsWithValuesInColumn(column='notes',
                                                   values=['fish', 'amplification', 'sish', 'case',
                                                           "method: if", "sanger", "sequencing"]),
                        DropRowsWithValuesInColumn(column='group', values=['histological type 2']),
                        DropRowsWithValuesInColumn(column='stain', values=['ctdna']),
                        DropRowsWithValuesInColumnExactMatch(column='notes', values=['LI']),

                        CalculateMissingValues(),

                        ConditionalOverWrite(
                            conditions=[
                                {
                                    'cond_field': 'expr_score',
                                    'cond': lambda expr_score: expr_score in ['0', 0, "-"],
                                    'res_field': 'expr_type',
                                    'res_value': 'negative'
                                },
                                {
                                    "cond_field": "expr_type",
                                    "cond": lambda expr_type: expr_type == "low",
                                    "res_field": "expr_intensity",
                                    "res_value": "low"
                                },
                                {
                                    "cond_field": "expr_type",
                                    "cond": lambda expr_type: expr_type == "high",
                                    "res_field": "expr_intensity",
                                    "res_value": "high"
                                },
                                {
                                    'cond_field': 'pattern',
                                    'cond': lambda pattern: pattern in PATTERN_INTO_PD_NAN,
                                    'res_field': 'pattern',
                                    'res_value': np.nan
                                }
                            ]
                        ),
                        MapExpressionTypes(key_words=['high', 'low'], goal_expr_type='positive'),

                        SelectHistologicalGroupFromTable(),

                        TumourMapperV3(
                            path_to_tumour_mappings=Path("../../data_retrieval/please_work_classification_final_v2/")),

                        DropRowsWithValuesInColumnExactMatch(column='matched_tumour',
                                                             values=['Artificial Taxon', "Not Ovarian"]),
                        DropNARowsInColumns(columns=['matched_tumour']),
                        enrich_df_with_whole_who_tree_classification,

                        DropIfNNAInRow(n=1, columns=['total_N', 'expr_n', 'expr_pct']),

                        MergeDifferingByColumnsPerTableID(differing_columns=['expr_score', 'expr_intensity',
                                                                             'expr_threshold', 'expr_distribution'],
                                                          do_remove=True),

                        FlipNegativeToPositive(),

                        CalculateExprPctAgain(),

                        DropDuplicates(mode="drop_all",
                                       # ignore={"table_id", "PT", "total_N", "expr_n", 'expr_pct'},
                                       ignore={"expr_intensity", "expr_threshold",
                                               "expr_score", "expr_distribution",
                                               "group", "group_cat", "PT",

                                               "total_N", "expr_n", 'expr_pct',

                                               "table_id", "tumour",
                                               'expr_type_old', 'stain_old', 'pattern_old'},
                                       differing_by_only={'table_id', 'tumour',
                                                          'expr_type_old', 'stain_old', 'pattern_old'},
                                       absolute_tolerances={
                                           'total_N': 2,
                                           'expr_n': 3
                                       },
                                       expr_pct_tolerance=2.5
                                       ),
                    ])

# concat = pd.concat([abstracts, tables], ignore_index=True).copy().reset_index(drop=True)
# concat.to_csv("raw.csv", index=False)

concat = pd.read_csv("../database_versions/raw.csv", low_memory=False).reset_index(drop=True)

NULL_STRINGS = {
    'null', 'none', 'na', 'n/a', 'nan',
    'n.a.', 'n.a', 'nil', 'missing',
    '', ' ',
}
for col in concat.columns:
    if concat[col].dtype == object:
        concat[col] = concat[col].apply(
            lambda x: np.nan if isinstance(x, str) and x.strip().lower() in NULL_STRINGS else x
        )

concat = concat.astype({'PMID': 'str', 'table_id': 'str'})

concat_raw = concat.copy(deep=True)

concat['PT'] = concat.PMID.map(PMID_TO_PT)

concat = pipeline(concat)
# concat.to_csv("../database_versions4/step6_stainology_database_updated.csv", index=False)
concat.shape

Rows dropped with values ['review', 'meta', 'systematic', 'editorial', 'letter', 'comment', 'news', 'interview', 'published erratum', 'retracted publication', 'retraction of publication'] in column: PT 1084
Rows dropped with values ['10427953_page_4_table_0', '9475192_page_5_table_0', '10427953_page_4_table_0', '29019869_page_5_table_0', '11567230_page_4_table_0', '29466768_page_3_table_0', '19033865_page_10_table_0', '25744580_page_4_table_0', '25074678_page_7_table_0', '23465277_page_3_table_0', '35621684_page_4_table_0', '27767100_page_3_table_0', '27767100_page_4_table_1', '11422476_abstract', '16499955_page_3_table_0', '26200099_abstract', '19033865_abstract', '10667156_page_1_table_0', '28594898_page_7_table_0'] in column: table_id 433
NA in stain: 0
NA in tumour: 28
NA in expr_type: 2007
NA in expr_n: 11988
Dropped rows with 2 or more NAs in columns ['total_N', 'expr_n', 'expr_pct']: 325
Removed rows with non-numeric signs in column expr_pct : 0
Removed rows with non-numeric sig

(10261, 26)